<a href="https://colab.research.google.com/github/PratikshaShind/OASIS-INFOBYTE/blob/main/PratikshaShinde_Task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Import Machine Learning tools from Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import NLTK components for advanced text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK corpora packages quietly
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Install and import WordCloud for structural visualization
!{sys.executable} -m pip install wordcloud --quiet
from wordcloud import WordCloud

print("💡 STEP 1: Upload your custom text CSV dataset...")
print("👉 Note: If you do not have one, click 'Cancel'. The script will auto-generate a sample dataset!")


uploaded = files.upload()

if len(uploaded) > 0:
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"\n[SUCCESS] '{filename}' loaded successfully!")
    # Auto-detect relevant text and sentiment columns if headers vary
    text_col = [c for c in df.columns if 'text' in c.lower() or 'review' in c.lower() or 'tweet' in c.lower() or 'comment' in c.lower()][0]
    label_col = [c for c in df.columns if 'sentiment' in c.lower() or 'label' in c.lower() or 'target' in c.lower()][0]
    df = df.rename(columns={text_col: 'Text', label_col: 'Sentiment'})
else:
    print("\n[INFO] No file uploaded. Generating synthetic sentiment dataset...")
    synthetic_data = {
        'Text': [
            "I absolutely love this product! It works perfectly and saved my time.",
            "Worst purchase ever. Broke on the first day and customer support was rude.",
            "It is an okay item. Nothing special, works as expected I guess.",
            "Amazing experience! Super fast delivery, highly recommend to everyone!",
            "Terrible quality. Complete waste of money. Avoid at all costs.",
            "The package arrived on time. The item quality is average.",
            "Fantastic service! The team was incredibly helpful and kind.",
            "I hate the new update. It crashes constantly and feels laggy.",
            "It serves its purpose. Neither good nor bad, just regular.",
            "Brilliant design, fits perfectly and looks beautiful!",
            "Defective item received. Disappointed with the packaging.",
            "An ordinary movie. Some parts were fine, others boring."
        ],
        'Sentiment': ['positive', 'negative', 'neutral', 'positive', 'negative', 'neutral',
                      'positive', 'negative', 'neutral', 'positive', 'negative', 'neutral']
    }
    df = pd.DataFrame(synthetic_data)

# Ensure uniform casing for label consistency
# Check if the 'Sentiment' column contains numerical values that need mapping to strings.
if pd.api.types.is_numeric_dtype(df['Sentiment']):
    print("⚠️ Detected numeric sentiment labels. Mapping to 'negative', 'neutral', 'positive'.")
    # Assuming 0: negative, 1: neutral, 2: positive based on common practices
    # and the number of classes in the synthetic data.
    sentiment_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
    df['Sentiment'] = df['Sentiment'].map(sentiment_map).astype(str)
df['Sentiment'] = df['Sentiment'].str.lower()


print("\n=== 📊 Sentiment Class Distribution ===")
dist = df['Sentiment'].value_counts()
print(dist)

# Plotting distribution bar chart
plt.figure(figsize=(7, 4))
sns.barplot(x=dist.index, y=dist.values, palette='viridis')
plt.title('Sentiment Distribution Overview')
plt.xlabel('Sentiment Class')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # 1. Lowercase conversions
    text = text.lower()
    # 2. Punctuation and Special Character removal
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Tokenization
    tokens = word_tokenize(text)
    # 4. Stopword filtering & Lemmatization
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(cleaned_tokens)

print("\n🔧 Processing text pipeline (Lowercasing, Punctuation removal, Stopwords removal, Lemmatization)...")
df['Clean_Text'] = df['Text'].apply(preprocess_text)
print("[SUCCESS] Processing completed.")


print("\n☁️ Generating Word Clouds per sentiment class...")
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
classes = ['positive', 'negative', 'neutral']

for i, sentiment in enumerate(classes):
    subset = df[df['Sentiment'] == sentiment]['Clean_Text']
    combined_text = " ".join(subset)

    if len(combined_text.strip()) > 0:
        wordcloud = WordCloud(width=400, height=300, background_color='white', max_words=30).generate(combined_text)
        axes[i].imshow(wordcloud, interpolation='bilinear')
        axes[i].set_title(f'{sentiment.upper()} Term Cloud', fontsize=14)
    else:
        axes[i].text(0.5, 0.5, 'Empty Text', horizontalalignment='center', verticalalignment='center')
    axes[i].axis('off')
plt.tight_layout()
plt.show()


print("\n🔄 Vectorizing features using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=2500)
X = vectorizer.fit_transform(df['Clean_Text']).toarray()
y = df['Sentiment']


# For very small synthetic frames, stratify parameter balances distribution splits evenly
X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
    X, y, df.index, test_size=0.20, random_state=42, stratify=y
)


models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

predictions = {}

for name, model in models.items():
    print(f"\n🚀 Training Model Engine: {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    # Structural Evaluation Evaluation
    print(f"--- {name} Performance Profile ---")
    print(f"Overall Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Matrix:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Plot Confusion Matrix mapping
    plt.figure(figsize=(4, 3))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=model.classes_, yticklabels=model.classes_)
    plt.title(f'{name} Confusion Matrix')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()


print("\n🔍 Conducting Error Analysis on Logistic Regression...")

# Create evaluation subset dataframe mapped from test indexes
test_df = df.loc[indices_test].copy()
test_df['Predicted_Sentiment'] = predictions["Logistic Regression"]

misclassified = test_df[test_df['Sentiment'] != test_df['Predicted_Sentiment']]

print(f"Total misclassified occurrences in test sample: {len(misclassified)}")
if len(misclassified) > 0:
    display(misclassified[['Text', 'Sentiment', 'Predicted_Sentiment']].head(5))
    """
    MARKDOWN CONCLUSION OF ERROR PROFILE:
    Models typically misclassify items due to:
    1. Short context length giving inadequate unique terms for mathematical variance.
    2. Shared non-descriptive terms appearing in both neutral and polarizing records.
    3. Tokenizing structures losing structural semantics (like sarcasm or inversion).
    """
else:
    print("✨ Perfect match achieved on test partitions! No misclassifications found to log.")

print("\n📝 === Executive Summary ===")
print("1. Performance: Logistic Regression typically handles multi-class micro-nuances better than Naive Bayes if data is dense.")

💡 STEP 1: Upload your custom text CSV dataset...
👉 Note: If you do not have one, click 'Cancel'. The script will auto-generate a sample dataset!
